In [0]:
print(spark.sql("SELECT current_catalog()").first()[0])   # expect: workspace
raw = spark.read.table("samples.nyctaxi.trips")
raw.printSchema()
display(raw.limit(5))

workspace
root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_zip: integer (nullable = true)
 |-- dropoff_zip: integer (nullable = true)



tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,1.31,7.5,10023,10023
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,1.8,9.5,10001,10018
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,2.3,11.5,10044,10111
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,2.6,18.5,10199,10022


In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.taxi_medallion")
spark.sql("USE workspace.taxi_medallion")

bronze = (spark.read.table("samples.nyctaxi.trips")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source", F.lit("samples.nyctaxi.trips")))

bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_trips")
print("bronze rows:", spark.table("bronze_trips").count())

bronze rows: 21932


In [0]:
%sql
DESCRIBE EXTENDED workspace.taxi_medallion.bronze_trips

col_name,data_type,comment
tpep_pickup_datetime,timestamp,null
tpep_dropoff_datetime,timestamp,null
trip_distance,double,null
fare_amount,double,null
pickup_zip,int,null
dropoff_zip,int,null
_ingested_at,timestamp,null
_source,string,null
,,
# Delta Statistics Columns,,


In [0]:
raw = spark.read.table("samples.nyctaxi.trips")
raw.printSchema()
display(raw.limit(5))

root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_zip: integer (nullable = true)
 |-- dropoff_zip: integer (nullable = true)



tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,1.31,7.5,10023,10023
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,1.8,9.5,10001,10018
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,2.3,11.5,10044,10111
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,2.6,18.5,10199,10022
